# Task 2.
## Implement a data loader for the BabyLM-Strict-Small data
- You will need a tokenizer (train a byte-level BPE Tokenizer from scratch)
- Take care to use all the sequences in the dataset (many inputs are longer than the maximum context length of the model you will train)

### Training a byte-level BPE Tokenizer from scratch 

**Notes:** By training a tokenizer on raw text we are turning strings into integer IDs. Here, it is important that the `vocab_size` of the model we've implemented so far matches that of the tokenizer's. Then, we encode all the BabyLM-Strict-Small data and use end-of-text tokens. Then, we seperate it into chunks of `max_length` tokens. This is smart, because nothing is discarded. Lastly, we wrap the chunks in a "data loader". This produces a format of `(inputs, targets, mask)`.

_The objective is is turn approx. 1.1M sentences into batches of `(inputs, targets, mask)`. Our GPT-2 model will then train on these batches._

#### 1. Inspecting the data

In [1]:
# Inspecting the data

from datasets import load_dataset

ds = load_dataset("BabyLM-community/BabyLM-2026-Strict-Small")
print(ds)

c:\Users\chris\anaconda3\envs\asds2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1104106
    })
})


It appears the dataset only has a `train` split, that is, 1M+ rows and one `text` column.

In [8]:
print(ds["train"][0:3])

{'text': ["Well it's just that, you know, a pound, or a hundred pounds today, is not the same as a hundred pounds in a year's time, or two, two years' time.", 'Right.', 'So that would be opportunity cost?']}


It looks like the first three sentences are about economics. Also, it appears we have one sentence per row.

#### 2. Train a byte-level BPE tokenizer

In [ ]:
from tokenizers import ByteLevelBPETokenizer

VOCAB_SIZE = 8000  # as states in the ATNLP-GPT hyperparameters

texts = ds["train"]["text"] # lists of strings, i.e., the text field of every row 

tokenizer = ByteLevelBPETokenizer() # empty byte-level BPE tokenizer
# "byte-level" means any text, because there are 256 bytes in the base alphabet

# training the empty tokenizer on BabyLM sentences
tokenizer.train_from_iterator(
    texts, # learn merges from this
    vocab_size=VOCAB_SIZE,
    min_frequency=2, # pairs are only merged if they are seen at least twice
    special_tokens=[
        "<pad>", # pads short sequences (ID 0)
        "<|endoftext|>"], # marking the end of each sentence (ID 1)
)

# save to current folder as .json and .txt files
tokenizer.save_model(".", "babylm_bpe") # then we do not have to retrain again

['.\\babylm_bpe-vocab.json', '.\\babylm_bpe-merges.txt']

**On BPE:** BPE begins with single bytes and groups common pairs into tokens (e.g. [32, 98, 121] -> 'm', 'a' and 'n'). When the vocabulary has reached 8000 tokens, it stops.

In [ ]:
# sanity check

print(tokenizer.get_vocab_size()) # must be 8000 as laid out in the assignment
print(tokenizer.token_to_id("<pad>"), tokenizer.token_to_id("<|endoftext|>")) # ID 0 and 1

# encoding test
enc_test = tokenizer.encode("There is a dog in the tree!")

# Note: BPE writes space using the "Ġ" symbol
print(enc_test.tokens) # how does it tokenize the sentences?
print(enc_test.ids) # what are each words' integer IDs?
print(tokenizer.decode(enc_test.ids)) # must return the original sentence

8000
0 1
['There', 'Ġis', 'Ġa', 'Ġdog', 'Ġin', 'Ġthe', 'Ġtree', '!']
[1041, 327, 260, 1212, 301, 263, 2180, 2]
There is a dog in the tree!


In [16]:
import torch

# we have two special tokens
eos_id = tokenizer.token_to_id("<|endoftext|>")
pad_id = tokenizer.token_to_id("<pad>")  

texts = [] # collecting all sentences
for row in ds["train"]:
    sentence = row["text"]
    if sentence.strip() != "": # skipping empty rows
        texts.append(sentence)

print(f"There are {len(texts)} sentences.") # should return approx. 1.1M sentences

There are 1104106 sentences.


**Empty rows:** There were no empty rows.

#### 3. Encode everything, making one long stream

E.g. the sentence "The man walks" could become the token IDs [12, 23, 102, 12]. All these are then stored into a single long tensor which we call `ids`. We save this is `babylm_ids.pt`.

In [ ]:
# encoding all 1.1M sentences at once (may take a while)

encodings = tokenizer.encode_batch(texts) # this gives us one encoding object per sentence
# sentences are still separable at this point! we just have a list of lists

all_ids = [] # we want just one (long) list of token IDs
# later we feed the stream into a "machine" that seperates it into 256-token chunks

for enc in encodings: # 1.1M iterations
    for token_id in enc.ids: # token IF of a given sentence
        all_ids.append(token_id)
    all_ids.append(eos_id) # mark with ID 1 (eos) after the sentence's tokens

# our model and dataset works with PyTorch tensors
ids = torch.tensor(all_ids, dtype=torch.long) # 64-bit integers

print(f"We have {ids.shape[0]} tokens all together.")

We have 17277036 tokens all together.


In [ ]:
# save the encodings
torch.save(ids, "babylm_ids.pt")

# this file is greater than 100MB and is stored locally on CSN's laptop

In [ ]:
print(ids[:10].tolist()) # first 10 token ids

# if we skip our special tokens, do we then see <|endoftext|>?
print(tokenizer.decode(ids[25:50].tolist(), skip_special_tokens=True)) 

print(ids.max().item() < 8000) # should be True; it finds the largest ID in the entire stream
# .item() turns a one-element tensor into a regular Python number
print((ids == eos_id).sum().item()) # should be 1104106, one per sentence

[824, 315, 309, 539, 321, 13, 291, 406, 13, 260]
 pounds in a year's time, or two, two years' time.<|endoftext|>Right.<|endoftext|>So that would be opportunity cost
True
1104106


#### 4. Divide the stream into chunks

In [ ]:
import math
import torch
from torch.utils.data import Dataset

IGNORE = -100 # something about cross entropy loss

# this class slices our long tensor into pieces of 256 tokens
# importantly, no token is discarded; long inputs are distributed across chunks
class PackedDataset(Dataset):
    def __init__(self, ids, block_size, pad_id):
        self.ids = ids # our tensor of token IDs
        self.block = block_size # we have 256 tokens per chunk (BPE)
        self.pad_id = pad_id # if the last chunk is short, we pad it (like putting it on a stool)

        # our first token can - by definition - not be a target, because it has nothing before it
        # hence, we will predict (N-1) tokens
        num_targets = len(ids) - 1
        # we need to round up; token leftovers get their own chunk
        self.num_chunks = math.ceil(num_targets/block_size)

    def __len__(self):
        # how many samples are there?
        return self.num_chunks

    def __getitem__(self, i):
        # the dataloader uses the following the retrieve sample i 
        start = i * self.block

        # use block + 1 tokens
        chunk = self.ids[start:start+self.block+1]

        x = chunk[:-1] # all except the last token can be used as inputs
        y = chunk[1:] # all except the first token can be used as targets

        # there is a posibility that the last chunk is too short
        # we pad it until it is a block
        if len(x) < self.block:
            n_pad = self.block - len(x)
            x = torch.cat([x, torch.full((n_pad,), self.pad_id)]) # WHAT DOES THIS DO??
            y = torch.cat([y, torch.full((n_pad,), IGNORE)])
        
        return x, y

#### 5. Batch and split

In [ ]:
# the dataloader, in its current form, receives 32 (x, y) pairs
# we make a function that merges the pairs into a single batch

# this function gather 32 chunks into one batch and builds a padding mask
def collate_fn(batch): # each sample gets one batch, which is just a list of (x, y) pairs
    all_x = []
    all_y = []
    for x, y in batch:
        all_x.append(x)
        all_y.append(y)

    x = torch.stack(all_x) # [batch_size, 256]
    y = torch.stack(all_y) # [batch_size, 256]

    mask = (x != pad_id).unsqueeze(1).unsqueeze(2) # the mask is True for real tokens and False for <pad>
    # we defined pad_id eariler
    # the shape is [B, 1, 1, T] - what is need for our MultiHeadAttention model

    return x, y, mask # WHAT HAPPENS HERE??

In [ ]:
from torch.utils.data import DataLoader, random_split

SEED = 42 # ATNLP-GPT hyperparameter
BLOCK = 256 # MAX_SEQUENCE_LENGTH
BATCH_SIZE = 32 # HUH??
VAL_FRACTION = 0.05

full_ds = PackedDataset(ids, BLOCK, pad_id)
# take the token stream and use it to build the dataset of chunks

# 5% of the chunks are randomly assigned to be used for validation, while the rest is used for training
n_val = int(VAL_FRACTION * len(full_ds))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

# at every epoch, shuffle the training data
# however, the validation data is not shuffled
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

#### 6. Sanity checks

In [33]:
# how many chunks do we have?
# it should be the number of tokens divided by 256
# also, the training dataset should take up 95% and validation the remaining

print(f"Tokens: {ids.shape[0]}")
print(f"Chunks: {len(full_ds)}")
print(f"Training chunks: {len(train_ds)}")
print(f"Validation chunks: {len(val_ds)}")

Tokens: 17277036
Chunks: 67489
Training chunks: 64115
Validation chunks: 3374


In [36]:
# what is the shape of our batches?

x, y, mask = next(iter(train_dl))

print(x.shape, y.shape, mask.shape)

torch.Size([32, 256]) torch.Size([32, 256]) torch.Size([32, 1, 1, 256])


In [ ]:
# also, targets should be the inputs, but shifted by 1
# the following should print True for this to be the case

print(torch.equal(x[0, 1:], y[0, :-1]))

True


In [39]:
# does the text look like normal?

print(tokenizer.decode(x[0].tolist(), skip_special_tokens=True)[:300])

CHI:	yeah.*MOT:	[laughs].[SI2 is counting some change]*INV:	can you read to me what you wrote Chi?*SI2:	five...[SI2 is still counting the change]*INV:	what did you write?*SI2:	six...*SI2:	seven.*INV:	what did you write honey?*CHI:	I don't wanna read it.[SIS is still counting the change]*INV:	oh come


**Note:** This looks rather strange

In [40]:
# does the number of IDs fit our vocabulary size of 8000?

print(x.max().item() < 8000, y.max().item() < 8000)

True True


In [ ]:
# is every token used just once, thereby living up to the "use all the sequences" requirement?

total_targets = 0
for i in range(len(full_ds)):
    x_i, y_i = full_ds[i] # MORE DETAILS
    total_targets += (y_i != IGNORE).sum().item() # do not count padded targets

print(total_targets, ids.shape[0]-1)

17277035 17277035


In [44]:
# is only the last chunk padded?

x_last, y_last = full_ds[len(full_ds)-1]
print(f"Padding of the last chunk: {(x_last == pad_id).sum().item()}")
print(f"We expect: {len(full_ds)*BLOCK-(ids.shape[0]-1)}")

Padding of the last chunk: 149
We expect: 149
